# Implement Feed-Forward Network from Scratch
### Problem Statement
The Feed-Forward Network (FFN) is the other main component in each Transformer layer (alongside attention). It's a simple **position-wise** MLP applied independently to each token.

The original Transformer uses a two-layer MLP with ReLU:
```
FFN(x) = W2 · ReLU(W1 · x + b1) + b2
```

Modern LLMs (Llama, Mistral, Gemma) use **SwiGLU**, a gated variant:
```
SwiGLU(x) = W2 · (Swish(W_gate · x) ⊙ (W_up · x))
```
where `Swish(x) = x · sigmoid(x)` and `⊙` is element-wise multiplication.

Your goal is to implement both variants from scratch.

---

### Requirements

1. **Standard FFN (ReLU)**
   - Two linear layers: `d_model → d_ff → d_model`
   - ReLU activation between them
   - Typically `d_ff = 4 * d_model`

2. **SwiGLU FFN**
   - Three linear layers: `W_gate`, `W_up` (both `d_model → d_ff`), `W_down` (`d_ff → d_model`)
   - Gate: `Swish(W_gate(x))` where `Swish(x) = x * sigmoid(x)` (same as `F.silu`)
   - Output: `W_down(Swish(W_gate(x)) * W_up(x))`

3. **Validate**
   - Output shape must be `(batch_size, seq_len, d_model)` for both.
   - Verify parameter counts match expectations.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Both variants must maintain the input/output dimension `d_model`.
- ✅ SwiGLU should use `F.silu` (Swish activation).

---

<details>
  <summary>💡 Hint</summary>

  - Standard FFN: `nn.Linear(d_model, d_ff)` → `F.relu()` → `nn.Linear(d_ff, d_model)`
  - SwiGLU has **3 weight matrices** instead of 2, but `d_ff` is often set to `(2/3) * 4 * d_model` to keep total params similar.
  - `F.silu(x)` is `x * torch.sigmoid(x)` — the Swish activation.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class FeedForward(nn.Module):
    """
    Standard Transformer FFN with ReLU activation.
    FFN(x) = W2 * ReLU(W1 * x + b1) + b2
    """
    def __init__(self, d_model: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        # TODO: Create two linear layers and dropout
        ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: W1 -> ReLU -> Dropout -> W2
        ...

In [ ]:
class SwiGLUFeedForward(nn.Module):
    """
    SwiGLU Feed-Forward Network used in Llama, Mistral, Gemma.
    SwiGLU(x) = W_down * (Swish(W_gate * x) * W_up * x)
    """
    def __init__(self, d_model: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        # SwiGLU typically uses (2/3) * 4 * d_model for d_ff
        # to keep param count similar to standard FFN
        d_ff = d_ff or int(2 / 3 * 4 * d_model)
        # TODO: Create W_gate, W_up (d_model -> d_ff), W_down (d_ff -> d_model), dropout
        ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Swish(W_gate(x)) * W_up(x) -> Dropout -> W_down
        ...

In [ ]:
# Test both FFN variants
torch.manual_seed(42)
d_model = 64
x = torch.randn(3, 4, d_model)  # (batch, seq_len, d_model)

ffn = FeedForward(d_model, dropout=0.0)
swiglu = SwiGLUFeedForward(d_model, dropout=0.0)

out_ffn = ffn(x)
out_swiglu = swiglu(x)

print(f"Input shape:       {x.shape}")
print(f"FFN output shape:  {out_ffn.shape}")
print(f"SwiGLU output:     {out_swiglu.shape}")
assert out_ffn.shape == x.shape
assert out_swiglu.shape == x.shape

# Compare parameter counts
ffn_params = sum(p.numel() for p in ffn.parameters())
swiglu_params = sum(p.numel() for p in swiglu.parameters())
print(f"\nFFN params:    {ffn_params:,}")
print(f"SwiGLU params: {swiglu_params:,}")
print("\n✅ Both FFN variants work correctly.")